# Prompt Caching: Paying Once for a Prefix

The full code of [Prompt Caching: Paying Once for a Prefix](https://residual-stream.lvtan71.workers.dev/posts/prompt-caching/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/prompt-caching) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/prompt-caching.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 3 min end to end.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q numpy "torch==2.14.*" "transformers==5.18.*" huggingface_hub

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "prompt-caching"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `breakeven.py`

The break-even of a 5-minute and a 1-hour cache from the published price multipliers, the bill for a 20k-token prefix sent 1,000 times, and the KV cache a provider holds for that prefix on three models.

In [ ]:
!{sys.executable} breakeven.py

The post printed:

```text
5-minute cache (write 1.25x, read 0.1x): cheaper from use 2 on; at 10 uses it costs 22% of uncached, at 100 uses 11%
1-hour cache (write 2.0x, read 0.1x): cheaper from use 3 on; at 10 uses it costs 29% of uncached, at 100 uses 12%

20k-token prefix, 1,000 requests: uncached $80.00, cached $4.10
KV cache for that 20k-token prefix, Qwen2.5-1.5B      :  0.53 GiB
KV cache for that 20k-token prefix, Qwen2.5-32B       :  4.88 GiB
KV cache for that 20k-token prefix, DeepSeek-R1 (MLA) :  1.31 GiB
```

## `ttft.py`

Time to the first token with and without a cached prefix, for prefixes of 1k to 8k tokens, on Qwen2.5-1.5B-Instruct.

In [ ]:
!{sys.executable} ttft.py

The post printed:

```text
KV cache per token: 28 KiB (2 x 28 layers x 2 KV heads x 128 x 2 bytes)
 prefix      cold    cached  saved  prefix KV held
   1024     918 ms      52 ms    94%          28 MiB
   2048    1843 ms      60 ms    97%          56 MiB
   4096    3854 ms      77 ms    98%         112 MiB
   8192    8819 ms     112 ms    99%         224 MiB
```